# Automated Web Security Scanner
**IBM SkillsBuild Cyber Security 2026**  
**Student:** Syeda Aairah Ahmed  
**Project level:** Beginner

## Project summary
This notebook reviews selected HTTP response security headers and highlights a few CORS response-header patterns for manual review. It provides an offline sample so the project can be demonstrated without contacting a website.

**Responsible use:** Only run a live check against a website you own or have explicit permission to assess. This is a basic educational checker, not a full vulnerability scanner. A missing header or a wildcard CORS value is a prompt for review, not proof of a vulnerability.

## 1. Requirements and setup
Install Python 3.10 or newer. Put this notebook and `requirements.txt` in the same folder, then run the following command in a terminal:

```bash
python -m pip install -r requirements.txt
```

If Jupyter is not already running, start it with:

```bash
jupyter notebook
```

Run the cells from top to bottom. The sample test below is offline and does not make any network requests.

In [ ]:
from urllib.parse import urlparse
from pathlib import Path

import pandas as pd
import requests
from requests.exceptions import RequestException

print("Setup successful: required libraries imported.")

## 2. Define the checks
The scanner checks for five common response headers and inspects the `Access-Control-Allow-Origin` CORS header. The results are intentionally simple and should be interpreted by a human reviewer.

In [ ]:
SECURITY_HEADERS = {
    "Content-Security-Policy": (
        "Missing CSP can leave browser content restrictions less explicit.",
        "Consider a Content Security Policy tailored to the site's resources."
    ),
    "Strict-Transport-Security": (
        "Missing HSTS means browsers may not be instructed to prefer HTTPS on future visits.",
        "Enable HSTS only after HTTPS is correctly configured for the domain and its subdomains as appropriate."
    ),
    "X-Content-Type-Options": (
        "Missing this header may allow MIME-type sniffing in some browser contexts.",
        "Consider setting X-Content-Type-Options: nosniff."
    ),
    "Referrer-Policy": (
        "Without a policy, referrer information may be shared according to browser defaults.",
        "Choose a Referrer-Policy appropriate to the application's privacy requirements."
    ),
    "Permissions-Policy": (
        "Without an explicit policy, browser feature access may be broader than intended.",
        "Consider restricting browser features the application does not need."
    ),
}


def analyze_headers(headers):
    """Analyze a mapping of response headers and return a pandas DataFrame."""
    # HTTP header names are case-insensitive.
    normalized = {str(key).lower(): str(value).strip() for key, value in headers.items()}
    findings = []

    for name, (reason, recommendation) in SECURITY_HEADERS.items():
        value = normalized.get(name.lower())
        if value:
            findings.append({
                "Check": name,
                "Status": "Present",
                "Observed value": value,
                "Recommendation": "Review the value to ensure it suits the application."
            })
        else:
            findings.append({
                "Check": name,
                "Status": "Missing",
                "Observed value": "Not returned",
                "Recommendation": recommendation
            })

    allow_origin = normalized.get("access-control-allow-origin")
    allow_credentials = normalized.get("access-control-allow-credentials", "").lower()

    if allow_origin == "*":
        findings.append({
            "Check": "CORS: Access-Control-Allow-Origin",
            "Status": "Review",
            "Observed value": allow_origin,
            "Recommendation": "Confirm wildcard access is intentional and appropriate for the resources being shared."
        })
    elif allow_origin:
        findings.append({
            "Check": "CORS: Access-Control-Allow-Origin",
            "Status": "Present",
            "Observed value": allow_origin,
            "Recommendation": "Confirm that only intended origins are allowed."
        })
    else:
        findings.append({
            "Check": "CORS: Access-Control-Allow-Origin",
            "Status": "Not observed",
            "Observed value": "Not returned",
            "Recommendation": "Review the relevant endpoints; CORS may be disabled or vary by route."
        })

    if allow_origin == "*" and allow_credentials == "true":
        findings.append({
            "Check": "CORS: credentials + wildcard",
            "Status": "Review",
            "Observed value": "wildcard origin and credentials=true",
            "Recommendation": "Review server configuration and actual browser behavior; do not rely on this header combination."
        })

    return pd.DataFrame(findings)

## 3. Run the offline sample
The sample headers are fictional. They are designed to demonstrate both present and missing headers plus a wildcard CORS value. No website is contacted.

In [ ]:
sample_headers = {
    "Content-Security-Policy": "default-src 'self'",
    "X-Content-Type-Options": "nosniff",
    "Access-Control-Allow-Origin": "*",
}

sample_results = analyze_headers(sample_headers)
display(sample_results)

print("\nStatus summary:")
display(sample_results["Status"].value_counts().rename_axis("Status").reset_index(name="Count"))

### Expected interpretation
- CSP and X-Content-Type-Options are marked **Present** because the sample includes them.
- HSTS, Referrer-Policy, and Permissions-Policy are marked **Missing** because the sample omits them.
- Wildcard CORS is marked **Review**. A wildcard is not automatically a vulnerability; its appropriateness depends on the resource and application context.

## 4. Optional live check — authorized targets only
This function makes one ordinary GET request, follows redirects, and applies a timeout. It does not crawl the site, exploit vulnerabilities, or test authentication. Do not run it against systems unless you own them or have explicit permission.

In [ ]:
def scan_authorized_url(url, timeout=10):
    """Fetch one URL and analyze its response headers.

    Only use on a system you own or have explicit permission to assess.
    """
    parsed = urlparse(url)
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("Enter a complete URL beginning with http:// or https://")

    try:
        response = requests.get(
            url,
            timeout=timeout,
            allow_redirects=True,
            headers={"User-Agent": "Educational-Web-Security-Header-Checker/1.0"},
        )
    except RequestException as exc:
        print(f"Request failed: {exc}")
        return None

    print(f"Final URL: {response.url}")
    print(f"HTTP status: {response.status_code}")
    print("Scope: one response-header check only; this is not a complete security audit.")
    return analyze_headers(response.headers)


# Example (intentionally commented out):
# authorized_url = "https://your-owned-domain.example"
# live_results = scan_authorized_url(authorized_url)
# if live_results is not None:
#     display(live_results)

## 5. Export the sample results
This writes a CSV file in the notebook's current working directory. You can include the CSV as supporting evidence if your submission portal permits extra files.

In [ ]:
output_path = Path("sample_scan_results.csv")
sample_results.to_csv(output_path, index=False)
print(f"Saved sample findings to: {output_path.resolve()}")

## 6. Limitations
- This project checks selected response headers only.
- It does not test authentication, authorization, application logic, server-side code, TLS quality, or all CORS behaviors.
- A header can be present but poorly configured; the scanner does not fully validate policy quality.
- Missing headers are review prompts, not automatic vulnerability findings.
- Results can vary by endpoint, redirect, user agent, and server configuration.

## 7. Conclusion
This project demonstrates beginner-level Python, HTTP response handling, basic security-header review, simple CORS awareness, tabular findings, and CSV export. Use it as a learning checklist and validate findings manually within an authorized scope.